# 🧊 Single Image → 3D Reconstruction (Research Notebook)

An end-to-end, reproducible research pipeline that turns **one photo** into a **textured 3D mesh** using a feed-forward Large Reconstruction Model ([TripoSR](https://github.com/VAST-AI-Research/TripoSR)), plus the tooling researchers usually have to write themselves:

| Stage | What you get |
|---|---|
| **Preprocessing** | Automatic background removal (or your own alpha), foreground re-framing |
| **Reconstruction** | Triplane NeRF encoding + marching-cubes mesh extraction at any resolution |
| **Post-processing** | Largest-component filtering, hole filling, Taubin smoothing, quadric decimation (colors preserved) |
| **Texturing** | Vertex colors or UV-baked texture atlas (xatlas) |
| **Visualisation** | Turntable GIF/MP4, contact sheet, interactive Plotly 3D viewer |
| **Evaluation** | Mesh statistics, Chamfer distance & F-score vs. a ground-truth mesh (with 24-orientation + ICP alignment), best-view SSIM/PSNR |
| **Experiments** | Batch processing to CSV, resolution × framing ablation sweeps, benchmark of time & VRAM |
| **Reproducibility** | Seeds, environment + git-commit logging, per-run `metadata.json` |
| **Demo** | Optional Gradio web app with a shareable link |

> **Runtime:** `Runtime → Change runtime type → T4 GPU` (or better). Then run cells top to bottom.

## 1 · Environment

In [ ]:
#@title 1.1 Check the runtime { display-mode: "form" }
import sys, platform
print("Python :", sys.version.split()[0], "|", platform.platform())
import torch
print("PyTorch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"GPU    : {p.name} | {p.total_memory/1e9:.1f} GB VRAM")
else:
    print("⚠️  No GPU detected. Use Runtime → Change runtime type → T4 GPU (CPU works but is very slow).")

In [ ]:
#@title 1.2 Install dependencies (≈2–4 min) { display-mode: "form" }
!pip install -q omegaconf einops trimesh rembg onnxruntime huggingface_hub "imageio[ffmpeg]" xatlas moderngl fast-simplification scikit-image scipy plotly gradio pandas matplotlib tqdm networkx 2>&1 | tail -n 3

import os, sys
if not os.path.isdir("TripoSR"):
    !git clone -q --depth 1 https://github.com/VAST-AI-Research/TripoSR.git
sys.path.insert(0, os.path.abspath("TripoSR"))
print("TripoSR source ready.")

In [ ]:
#@title 1.3 Marching-cubes backend (auto-fallback) { display-mode: "form" }
# TripoSR extracts meshes with `torchmcubes`, which compiles against your CUDA/PyTorch.
# If the build fails we install a tiny scikit-image based shim so the notebook still works.
import importlib, textwrap, os, sys

def _has_mc():
    try:
        importlib.import_module("torchmcubes"); return True
    except Exception:
        return False

if not _has_mc():
    !pip install -q git+https://github.com/tatsy/torchmcubes.git 2>&1 | tail -n 3
    importlib.invalidate_caches()

MC_BACKEND = "torchmcubes" if _has_mc() else None
if MC_BACKEND is None:
    os.makedirs("shim", exist_ok=True)
    with open("shim/torchmcubes.py", "w") as f:
        f.write(textwrap.dedent("""
            import numpy as np, torch
            from skimage import measure
            def marching_cubes(vol, thresh):
                v = vol.detach().float().cpu().numpy()
                try:
                    verts, faces, _, _ = measure.marching_cubes(v, level=float(thresh))
                except (ValueError, RuntimeError):
                    return (torch.zeros((0, 3), device=vol.device),
                            torch.zeros((0, 3), dtype=torch.long, device=vol.device))
                verts = np.ascontiguousarray(verts[:, ::-1])  # match torchmcubes axis order
                return (torch.from_numpy(verts).float().to(vol.device),
                        torch.from_numpy(faces.astype(np.int64)).to(vol.device))
        """))
    sys.path.insert(0, os.path.abspath("shim"))
    MC_BACKEND = "skimage-fallback"
print("Marching-cubes backend:", MC_BACKEND)
if MC_BACKEND != "torchmcubes":
    print("ℹ️  Fallback is slower and its axis convention is verified visually: check the 3D viewer orientation.")

## 2 · Configuration

All experiment knobs live in one dataclass, so every run is fully described by its saved `metadata.json`.

In [ ]:
#@title 2.1 Experiment configuration { display-mode: "form" }
import os, sys, io, json, time, math, glob, gc, hashlib, platform, zipfile, random, itertools, subprocess, warnings
from dataclasses import dataclass, asdict, replace
from pathlib import Path
import numpy as np
from PIL import Image
import torch, trimesh
warnings.filterwarnings("ignore")

#@markdown ### Reproducibility
SEED = 42 #@param {type:"integer"}
#@markdown ### Preprocessing
REMOVE_BACKGROUND = True #@param {type:"boolean"}
FOREGROUND_RATIO = 0.85 #@param {type:"slider", min:0.5, max:1.0, step:0.01}
#@markdown ### Reconstruction
MC_RESOLUTION = 256 #@param [128, 192, 256, 320, 384, 512] {type:"raw"}
MC_THRESHOLD = 25.0 #@param {type:"number"}
CHUNK_SIZE = 8192 #@param [2048, 4096, 8192, 16384] {type:"raw"}
VERTEX_COLORS = True #@param {type:"boolean"}
#@markdown ### Mesh post-processing
KEEP_LARGEST_COMPONENT = True #@param {type:"boolean"}
FILL_HOLES = True #@param {type:"boolean"}
TAUBIN_SMOOTH_ITERS = 0 #@param {type:"integer"}
DECIMATE_TO_FACES = 0 #@param {type:"integer"}
#@markdown ### Texture / rendering / export
BAKE_TEXTURE = False #@param {type:"boolean"}
TEXTURE_RESOLUTION = 2048 #@param [512, 1024, 2048, 4096] {type:"raw"}
RENDER_VIEWS = 36 #@param {type:"integer"}
RENDER_ELEVATION_DEG = 15 #@param {type:"slider", min:-30, max:60, step:5}
VIEWER_ORIENTATION = True #@param {type:"boolean"}
OUTPUT_DIR = "outputs" #@param {type:"string"}

@dataclass
class Config:
    seed: int = SEED
    remove_background: bool = REMOVE_BACKGROUND
    foreground_ratio: float = FOREGROUND_RATIO
    mc_resolution: int = MC_RESOLUTION
    mc_threshold: float = MC_THRESHOLD
    chunk_size: int = CHUNK_SIZE
    vertex_colors: bool = VERTEX_COLORS
    keep_largest_component: bool = KEEP_LARGEST_COMPONENT
    fill_holes: bool = FILL_HOLES
    taubin_smooth_iters: int = TAUBIN_SMOOTH_ITERS
    decimate_to_faces: int = DECIMATE_TO_FACES
    bake_texture: bool = BAKE_TEXTURE
    texture_resolution: int = TEXTURE_RESOLUTION
    render_views: int = RENDER_VIEWS
    render_elevation_deg: float = RENDER_ELEVATION_DEG
    viewer_orientation: bool = VIEWER_ORIENTATION
    output_dir: str = OUTPUT_DIR

cfg = Config()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def set_seed(seed: int):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

set_seed(cfg.seed)
print(json.dumps(asdict(cfg), indent=2))

In [ ]:
#@title 2.2 (Optional) Mount Google Drive to persist results { display-mode: "form" }
MOUNT_DRIVE = False #@param {type:"boolean"}
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/img2mesh_outputs" #@param {type:"string"}
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
    cfg = replace(cfg, output_dir=DRIVE_OUTPUT_DIR)
    print("Outputs will be written to:", cfg.output_dir)

## 3 · Load the model

In [ ]:
#@title 3.1 Load TripoSR weights from Hugging Face { display-mode: "form" }
from tsr.system import TSR
from tsr.utils import remove_background, resize_foreground

t0 = time.time()
model = TSR.from_pretrained("stabilityai/TripoSR", config_name="config.yaml", weight_name="model.ckpt")
model.renderer.set_chunk_size(cfg.chunk_size)
model.to(DEVICE)
model.eval()
n_params = sum(p.numel() for p in model.parameters())
print(f"Loaded TripoSR on {DEVICE} in {time.time()-t0:.1f}s | {n_params/1e6:.1f}M parameters")

## 4 · Core library

Modular functions (preprocess → encode → extract → post-process → bake → render → evaluate). They are reused by the single-image run, the batch runner, the ablation sweep and the web demo, so you can swap in your own steps easily.

In [ ]:
#@title 4.1 Preprocessing { display-mode: "form" }
_rembg_session = None
def get_rembg():
    global _rembg_session
    if _rembg_session is None:
        import rembg
        _rembg_session = rembg.new_session()
    return _rembg_session

def has_useful_alpha(img: Image.Image) -> bool:
    if img.mode != "RGBA":
        return False
    a = np.asarray(img)[..., 3]
    return a.min() < 250 and a.max() > 5

def preprocess(img: Image.Image, remove_bg=True, ratio=0.85):
    """Returns (processed_rgb_on_gray, rgba_or_None).
    - If the input already has a meaningful alpha channel it is used as the mask.
    - Else, if remove_bg, rembg (U²-Net) segments the object.
    - The object is re-framed to `ratio` of the canvas and composited on 50% gray (TripoSR's training convention).
    """
    rgba = None
    if has_useful_alpha(img):
        rgba = img
    elif remove_bg:
        rgba = remove_background(img.convert("RGB"), get_rembg())
    if rgba is not None:
        rgba = resize_foreground(rgba, ratio)
        arr = np.asarray(rgba).astype(np.float32) / 255.0
        arr = arr[..., :3] * arr[..., 3:4] + (1.0 - arr[..., 3:4]) * 0.5
        return Image.fromarray((arr * 255.0).astype(np.uint8)), rgba
    return img.convert("RGB"), None

@torch.no_grad()
def encode(image: Image.Image):
    """Image → triplane scene code. Returns (scene_codes, seconds)."""
    t = time.time()
    scene_codes = model(image, device=DEVICE)
    if DEVICE == "cuda": torch.cuda.synchronize()
    return scene_codes, time.time() - t

In [ ]:
#@title 4.2 Mesh extraction, post-processing and texture baking { display-mode: "form" }
from scipy.spatial import cKDTree

@torch.no_grad()
def extract_mesh(scene_codes, resolution=256, threshold=25.0, vertex_colors=True):
    try:
        mesh = model.extract_mesh(scene_codes, vertex_colors, resolution=resolution, threshold=threshold)[0]
    except TypeError:  # older TripoSR revisions have no `threshold` argument
        mesh = model.extract_mesh(scene_codes, vertex_colors, resolution=resolution)[0]
    return mesh

def _transfer_vertex_colors(src, dst):
    try:
        cols = np.asarray(src.visual.vertex_colors)
        _, idx = cKDTree(np.asarray(src.vertices)).query(np.asarray(dst.vertices))
        dst.visual = trimesh.visual.ColorVisuals(mesh=dst, vertex_colors=cols[idx])
    except Exception as e:
        print("  (color transfer skipped:", e, ")")
    return dst

def _largest_component(m):
    parts = m.split(only_watertight=False)
    return max(parts, key=lambda p: len(p.faces)) if len(parts) > 1 else m

def _decimate(m, target):
    if target <= 0 or len(m.faces) <= target:
        return m
    try:
        d = m.simplify_quadric_decimation(face_count=target)
    except TypeError:
        d = m.simplify_quadric_decimation(target)
    return _transfer_vertex_colors(m, d)

def postprocess(mesh, c: Config):
    m = mesh.copy()
    m.update_faces(m.nondegenerate_faces())
    m.remove_unreferenced_vertices()
    if c.keep_largest_component:
        m = _largest_component(m)
    if c.fill_holes:
        try: trimesh.repair.fill_holes(m)
        except Exception as e: print("  (fill_holes skipped:", e, ")")
    try:
        trimesh.repair.fix_normals(m)
    except Exception as e:
        print("  (fix_normals skipped:", e, ")")
    if c.taubin_smooth_iters > 0:
        trimesh.smoothing.filter_taubin(m, lamb=0.5, nu=0.53, iterations=c.taubin_smooth_iters)
    m = _decimate(m, c.decimate_to_faces)
    return m

def orient_for_viewers(mesh):
    """Rotate from TripoSR's native frame to the Y-up convention used by GLB viewers."""
    m = mesh.copy()
    m.apply_transform(trimesh.transformations.rotation_matrix(-np.pi / 2, [1, 0, 0]))
    m.apply_transform(trimesh.transformations.rotation_matrix(np.pi / 2, [0, 1, 0]))
    return m

def bake_texture_atlas(mesh, scene_code, texture_resolution, out_dir: Path):
    """UV-unwrap with xatlas and bake colors from the NeRF into a PNG atlas. Needs a headless GL context."""
    from tsr.bake_texture import bake_texture
    import xatlas
    out = bake_texture(mesh, model, scene_code, texture_resolution)
    obj_path = out_dir / "mesh_textured.obj"
    xatlas.export(str(obj_path), mesh.vertices[out["vmapping"]], out["indices"], out["uvs"],
                  mesh.vertex_normals[out["vmapping"]])
    tex = Image.fromarray((out["colors"] * 255.0).astype(np.uint8)).transpose(Image.FLIP_TOP_BOTTOM)
    tex.save(out_dir / "texture.png")
    return str(obj_path), str(out_dir / "texture.png")

In [ ]:
#@title 4.3 Rendering and interactive viewers { display-mode: "form" }
import imageio, base64
import matplotlib.pyplot as plt
from IPython.display import HTML, display

@torch.no_grad()
def render_turntable(scene_codes, n_views=36, elevation_deg=15.0, size=384):
    """Renders orbit views directly from the NeRF (not the mesh)."""
    try:
        frames = model.render(scene_codes, n_views=n_views, elevation_deg=elevation_deg,
                              height=size, width=size, return_type="pil")[0]
    except TypeError:
        frames = model.render(scene_codes, n_views=n_views, return_type="pil")[0]
    return frames

def save_turntable(frames, out_dir: Path, fps=15):
    arrs = [np.asarray(f.convert("RGB")) for f in frames]
    gif, mp4 = out_dir / "turntable.gif", out_dir / "turntable.mp4"
    imageio.mimsave(gif, arrs, duration=1000 / fps, loop=0)
    try:
        imageio.mimsave(mp4, arrs, fps=fps, macro_block_size=1)
    except Exception as e:
        print("  (mp4 export skipped:", e, ")"); mp4 = None
    return str(gif), (str(mp4) if mp4 else None)

def show_video(path, width=360):
    b64 = base64.b64encode(open(path, "rb").read()).decode()
    display(HTML(f'<video width="{width}" controls autoplay loop muted><source src="data:video/mp4;base64,{b64}" type="video/mp4"></video>'))

def contact_sheet(frames, n=8, out_path=None, title=None):
    idx = np.linspace(0, len(frames) - 1, n, endpoint=False).astype(int)
    cols = min(n, 4); rows = math.ceil(n / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(3 * cols, 3 * rows))
    for ax in np.atleast_1d(axes).ravel(): ax.axis("off")
    for ax, i in zip(np.atleast_1d(axes).ravel(), idx):
        ax.imshow(frames[i]); ax.set_title(f"view {i}", fontsize=9)
    if title: fig.suptitle(title)
    plt.tight_layout()
    if out_path: fig.savefig(out_path, dpi=120)
    plt.show()

def show_plotly(mesh, max_faces=60000, title="Reconstructed mesh"):
    import plotly.graph_objects as go
    m = _decimate(mesh, max_faces) if len(mesh.faces) > max_faces else mesh
    v, f = np.asarray(m.vertices), np.asarray(m.faces)
    kw = {}
    try:
        cols = np.asarray(m.visual.vertex_colors)[:, :3]
        kw["vertexcolor"] = cols
    except Exception:
        kw["color"] = "lightgray"
    fig = go.Figure(go.Mesh3d(x=v[:, 0], y=v[:, 1], z=v[:, 2], i=f[:, 0], j=f[:, 1], k=f[:, 2],
                              lighting=dict(ambient=0.6, diffuse=0.8, specular=0.2), **kw))
    fig.update_layout(title=title, scene=dict(aspectmode="data"), height=560, margin=dict(l=0, r=0, t=40, b=0))
    fig.show()

In [ ]:
#@title 4.4 Evaluation metrics { display-mode: "form" }
def mesh_stats(m: trimesh.Trimesh) -> dict:
    s = {
        "vertices": int(len(m.vertices)), "faces": int(len(m.faces)),
        "watertight": bool(m.is_watertight), "winding_consistent": bool(m.is_winding_consistent),
        "euler_number": int(m.euler_number), "surface_area": float(m.area),
        "extents": [float(x) for x in m.extents],
    }
    try: s["components"] = int(len(m.split(only_watertight=False)))
    except Exception: s["components"] = None
    s["volume"] = float(m.volume) if m.is_watertight else None
    return s

def _norm_pts(pts):
    c = (pts.max(0) + pts.min(0)) / 2.0
    s = (pts.max(0) - pts.min(0)).max()
    return (pts - c) / max(s, 1e-9), c, s

def _rot24():
    mats = []
    for perm in itertools.permutations(range(3)):
        for signs in itertools.product([1, -1], repeat=3):
            M = np.zeros((3, 3))
            for i, (p, sg) in enumerate(zip(perm, signs)): M[i, p] = sg
            if np.isclose(np.linalg.det(M), 1.0): mats.append(M)
    return mats
ROT24 = _rot24()

def _sample(m, n):
    pts, _ = trimesh.sample.sample_surface(m, n)
    return np.asarray(pts)

def chamfer_fscore(pred: trimesh.Trimesh, gt: trimesh.Trimesh, n=100000, taus=(0.01, 0.02, 0.05),
                   align=True, seed=0):
    """Chamfer-L2 / L1 and F-score@τ on unit-normalised meshes.
    With align=True the prediction is aligned to GT by trying all 24 axis-aligned rotations, then ICP.
    τ is expressed as a fraction of the GT bounding-box max extent."""
    np.random.seed(seed)
    pa, pb = _sample(pred, n), _sample(gt, n)
    pa, _, _ = _norm_pts(pa); pb, _, _ = _norm_pts(pb)
    best_R = np.eye(3)
    if align:
        tb = cKDTree(pb[:: max(1, n // 4000)])
        sub = pa[:: max(1, n // 4000)]
        costs = []
        for R in ROT24:
            q = sub @ R.T
            costs.append(tb.query(q)[0].mean() + cKDTree(q).query(pb[:: max(1, n // 4000)])[0].mean())
        best_R = ROT24[int(np.argmin(costs))]
        pa = pa @ best_R.T
        try:
            T, _, _ = trimesh.registration.icp(pa[::10], pb[::10], max_iterations=50, scale=False)
            pa = trimesh.transform_points(pa, T)
        except Exception as e:
            print("  (ICP skipped:", e, ")")
    d_ab = cKDTree(pb).query(pa)[0]; d_ba = cKDTree(pa).query(pb)[0]
    out = {"chamfer_l2": float((d_ab ** 2).mean() + (d_ba ** 2).mean()),
           "chamfer_l1": float(d_ab.mean() + d_ba.mean()),
           "alignment_rotation": best_R.tolist()}
    for tau in taus:
        p, r = float((d_ab < tau).mean()), float((d_ba < tau).mean())
        out[f"fscore@{tau}"] = 0.0 if p + r == 0 else 2 * p * r / (p + r)
        out[f"precision@{tau}"], out[f"recall@{tau}"] = p, r
    return out

def best_view_fidelity(processed: Image.Image, frames):
    """Approximate reference-view fidelity: best SSIM / PSNR between the processed input and any rendered orbit view.
    (Camera conventions differ from your photo, so treat as a relative signal for comparing settings.)"""
    from skimage.metrics import structural_similarity, peak_signal_noise_ratio
    ref = np.asarray(processed.convert("RGB").resize((256, 256))).astype(np.float32) / 255.0
    best = {"ssim": -1.0, "psnr": -1.0, "view": -1}
    for i, f in enumerate(frames):
        x = np.asarray(f.convert("RGB").resize((256, 256))).astype(np.float32) / 255.0
        s = structural_similarity(ref, x, channel_axis=2, data_range=1.0)
        if s > best["ssim"]:
            best = {"ssim": float(s), "psnr": float(peak_signal_noise_ratio(ref, x, data_range=1.0)), "view": i}
    return best

In [ ]:
#@title 4.5 Full pipeline runner (with run logging) { display-mode: "form" }
def _sha256(img: Image.Image) -> str:
    return hashlib.sha256(img.convert("RGBA").tobytes()).hexdigest()

def _git_commit(path="TripoSR"):
    try: return subprocess.check_output(["git", "-C", path, "rev-parse", "HEAD"], text=True).strip()
    except Exception: return None

def environment_info():
    import transformers
    return {"python": sys.version.split()[0], "platform": platform.platform(),
            "torch": torch.__version__, "cuda": torch.version.cuda, "trimesh": trimesh.__version__,
            "transformers": transformers.__version__, "numpy": np.__version__,
            "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
            "triposr_commit": _git_commit(), "marching_cubes_backend": MC_BACKEND}

def export_mesh(mesh, out_dir: Path, orient=True):
    m = orient_for_viewers(mesh) if orient else mesh
    files = {}
    for ext in ["glb", "obj", "ply", "stl"]:
        p = out_dir / f"mesh.{ext}"
        try:
            m.export(p); files[ext] = str(p)
        except Exception as e:
            print(f"  ({ext} export skipped: {e})")
    return files

def run_pipeline(image: Image.Image, c: Config, name="sample", reference_mesh=None, save=True):
    set_seed(c.seed)
    model.renderer.set_chunk_size(c.chunk_size)
    out_dir = Path(c.output_dir) / f"{name}_{time.strftime('%Y%m%d_%H%M%S')}"
    out_dir.mkdir(parents=True, exist_ok=True)
    if DEVICE == "cuda":
        torch.cuda.reset_peak_memory_stats(); torch.cuda.empty_cache()
    T = {}

    t = time.time()
    processed, rgba = preprocess(image, c.remove_background, c.foreground_ratio)
    T["preprocess_s"] = time.time() - t
    processed.save(out_dir / "input_processed.png")
    image.convert("RGB").save(out_dir / "input_original.png")

    scene_codes, T["encode_s"] = encode(processed)

    t = time.time()
    raw = extract_mesh(scene_codes, c.mc_resolution, c.mc_threshold, c.vertex_colors)
    T["extract_s"] = time.time() - t
    if len(raw.faces) == 0:
        raise RuntimeError("Empty mesh. Try lowering mc_threshold, enabling background removal, or raising foreground_ratio.")

    t = time.time()
    mesh = postprocess(raw, c)
    T["postprocess_s"] = time.time() - t

    files = {}
    if save:
        files.update(export_mesh(mesh, out_dir, orient=c.viewer_orientation))
    if c.bake_texture and save:
        try:
            t = time.time()
            obj, tex = bake_texture_atlas(raw, scene_codes[0], c.texture_resolution, out_dir)
            files["textured_obj"], files["texture_png"] = obj, tex
            T["bake_s"] = time.time() - t
        except Exception as e:
            print(f"⚠️  Texture baking failed ({type(e).__name__}: {e}). Vertex-color meshes were still exported.")

    frames = None
    if c.render_views > 0:
        t = time.time()
        frames = render_turntable(scene_codes, c.render_views, c.render_elevation_deg)
        T["render_s"] = time.time() - t
        if save:
            files["turntable_gif"], files["turntable_mp4"] = save_turntable(frames, out_dir)

    metrics = {"mesh": mesh_stats(mesh), "raw_mesh": {"vertices": len(raw.vertices), "faces": len(raw.faces)}}
    if frames is not None:
        metrics["best_view_fidelity"] = best_view_fidelity(processed, frames)
    if reference_mesh is not None:
        metrics["vs_reference"] = chamfer_fscore(mesh, reference_mesh)
    T["total_s"] = sum(T.values())
    if DEVICE == "cuda":
        T["peak_vram_gb"] = torch.cuda.max_memory_allocated() / 1e9

    meta = {"name": name, "config": asdict(c), "input_sha256": _sha256(image), "timings": T,
            "metrics": metrics, "environment": environment_info(), "files": files}
    if save:
        with open(out_dir / "metadata.json", "w") as f: json.dump(meta, f, indent=2, default=str)
    return {"out_dir": str(out_dir), "processed": processed, "rgba": rgba, "scene_codes": scene_codes,
            "raw_mesh": raw, "mesh": mesh, "frames": frames, "files": files, "metadata": meta}

def show_result(res):
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(1, 2, figsize=(8, 4))
    for a, im, t in zip(ax, [res["metadata"].get("_orig") or res["processed"], res["processed"]], ["input", "processed"]):
        a.imshow(im); a.set_title(t); a.axis("off")
    plt.show()
    md_ = res["metadata"]
    print(json.dumps({"timings": md_["timings"], "metrics": md_["metrics"]}, indent=2, default=str))
    if res["frames"] is not None:
        contact_sheet(res["frames"], n=8, title="NeRF orbit renders")
        mp4 = res["files"].get("turntable_mp4")
        if mp4: show_video(mp4)
    show_plotly(orient_for_viewers(res["mesh"]) if md_["config"]["viewer_orientation"] else res["mesh"])
    print("Saved to:", res["out_dir"])

## 5 · Choose an input image

Tips: one centred object, uncluttered background, even lighting, no strong perspective distortion, object fully visible.

In [ ]:
#@title 5.1 Load the image { display-mode: "form" }
import requests
INPUT_SOURCE = "sample" #@param ["sample", "upload", "url", "path"]
SAMPLE_INDEX = 0 #@param {type:"integer"}
IMAGE_URL = "" #@param {type:"string"}
IMAGE_PATH = "" #@param {type:"string"}

def load_input():
    if INPUT_SOURCE == "sample":
        files = sorted(glob.glob("TripoSR/examples/*.png") + glob.glob("TripoSR/examples/*.jpg"))
        p = files[SAMPLE_INDEX % len(files)]
        return Image.open(p), Path(p).stem
    if INPUT_SOURCE == "upload":
        from google.colab import files as colab_files
        up = colab_files.upload()
        p = next(iter(up)); return Image.open(p), Path(p).stem
    if INPUT_SOURCE == "url":
        r = requests.get(IMAGE_URL, headers={"User-Agent": "Mozilla/5.0"}, timeout=30); r.raise_for_status()
        return Image.open(io.BytesIO(r.content)), "url_image"
    return Image.open(IMAGE_PATH), Path(IMAGE_PATH).stem

input_image, input_name = load_input()
print(f"Loaded '{input_name}' | size={input_image.size} | mode={input_image.mode}")
display(input_image.convert("RGBA").resize((min(384, input_image.width), int(min(384, input_image.width) * input_image.height / input_image.width))))

## 6 · Reconstruct

Runs the full pipeline and shows the preprocessing result, orbit renders, turntable video and an interactive 3D viewer.

In [ ]:
#@title 6.1 Run single-image reconstruction { display-mode: "form" }
result = run_pipeline(input_image, cfg, name=input_name)
result["metadata"]["_orig"] = input_image.convert("RGB")
show_result(result)

### 6.2 (Optional) Evaluate against a ground-truth mesh

Provide a `.obj/.glb/.ply` ground-truth file to compute **Chamfer distance** and **F-score@τ** (τ as a fraction of the object's size). The prediction is automatically aligned to the ground truth (24 axis-aligned rotations + ICP). Scale is normalised, so units don't matter.

In [ ]:
#@title 6.2 Compare with a ground-truth mesh { display-mode: "form" }
GT_SOURCE = "none" #@param ["none", "upload", "path"]
GT_PATH = "" #@param {type:"string"}
gt = None
if GT_SOURCE == "upload":
    from google.colab import files as colab_files
    up = colab_files.upload(); GT_PATH = next(iter(up))
if GT_SOURCE != "none":
    gt = trimesh.load(GT_PATH, force="mesh")
    scores = chamfer_fscore(result["mesh"], gt)
    print(json.dumps({k: v for k, v in scores.items() if k != "alignment_rotation"}, indent=2))
    result["metadata"]["metrics"]["vs_reference"] = scores
    with open(Path(result["out_dir"]) / "metadata.json", "w") as f:
        json.dump({k: v for k, v in result["metadata"].items() if k != "_orig"}, f, indent=2, default=str)
else:
    print("Skipped (no ground-truth mesh selected).")

## 7 · Experiments

In [ ]:
#@title 7.1 Batch-process a folder of images → CSV benchmark { display-mode: "form" }
import pandas as pd
from tqdm.auto import tqdm
BATCH_FOLDER = "TripoSR/examples" #@param {type:"string"}
BATCH_LIMIT = 6 #@param {type:"integer"}
RUN_BATCH = False #@param {type:"boolean"}

def run_batch(folder, c: Config, limit=None):
    paths = sorted([p for ext in ("png", "jpg", "jpeg", "webp") for p in glob.glob(f"{folder}/*.{ext}")])
    paths = paths[:limit] if limit else paths
    rows = []
    for p in tqdm(paths):
        try:
            res = run_pipeline(Image.open(p), replace(c, render_views=0), name=Path(p).stem)
            m, T = res["metadata"]["metrics"]["mesh"], res["metadata"]["timings"]
            rows.append({"image": Path(p).name, "faces": m["faces"], "watertight": m["watertight"],
                         "components": m["components"], **T, "out_dir": res["out_dir"]})
        except Exception as e:
            rows.append({"image": Path(p).name, "error": f"{type(e).__name__}: {e}"})
        gc.collect(); torch.cuda.empty_cache() if DEVICE == "cuda" else None
    return pd.DataFrame(rows)

if RUN_BATCH:
    batch_df = run_batch(BATCH_FOLDER, cfg, BATCH_LIMIT)
    csv_path = Path(cfg.output_dir) / "batch_results.csv"; batch_df.to_csv(csv_path, index=False)
    display(batch_df); print("Saved:", csv_path)
else:
    print("Tick RUN_BATCH to execute.")

In [ ]:
#@title 7.2 Ablation sweep: foreground ratio × marching-cubes resolution { display-mode: "form" }
import pandas as pd
RUN_ABLATION = False #@param {type:"boolean"}
ABL_RATIOS = "0.7, 0.85, 0.95" #@param {type:"string"}
ABL_RESOLUTIONS = "128, 256, 384" #@param {type:"string"}

def run_ablation(image, ratios, resolutions, c: Config):
    """Encodes once per framing ratio and re-extracts the mesh at every resolution (cheap)."""
    rows = []
    for r in ratios:
        set_seed(c.seed)
        processed, _ = preprocess(image, c.remove_background, r)
        scene_codes, t_enc = encode(processed)
        for res in resolutions:
            t = time.time(); raw = extract_mesh(scene_codes, res, c.mc_threshold, c.vertex_colors); t_ex = time.time() - t
            if len(raw.faces) == 0:
                rows.append({"foreground_ratio": r, "mc_resolution": res, "faces": 0}); continue
            m = postprocess(raw, c); s = mesh_stats(m)
            rows.append({"foreground_ratio": r, "mc_resolution": res, "encode_s": t_enc, "extract_s": t_ex,
                         "faces": s["faces"], "watertight": s["watertight"], "components": s["components"],
                         "surface_area": s["surface_area"]})
    return pd.DataFrame(rows)

if RUN_ABLATION:
    ratios = [float(x) for x in ABL_RATIOS.split(",")]
    resolutions = [int(x) for x in ABL_RESOLUTIONS.split(",")]
    abl = run_ablation(input_image, ratios, resolutions, cfg)
    display(abl)
    abl.to_csv(Path(cfg.output_dir) / "ablation.csv", index=False)
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    for r, g in abl.groupby("foreground_ratio"):
        ax[0].plot(g["mc_resolution"], g["faces"], marker="o", label=f"ratio {r}")
        ax[1].plot(g["mc_resolution"], g["extract_s"], marker="o", label=f"ratio {r}")
    ax[0].set(xlabel="MC resolution", ylabel="faces", title="Mesh complexity"); ax[1].set(xlabel="MC resolution", ylabel="seconds", title="Extraction time")
    for a in ax: a.legend(); a.grid(alpha=.3)
    plt.tight_layout(); plt.show()
else:
    print("Tick RUN_ABLATION to execute.")

In [ ]:
#@title 7.3 Compare reconstructions across thresholds (side-by-side renders) { display-mode: "form" }
RUN_THRESHOLD_SWEEP = False #@param {type:"boolean"}
THRESHOLDS = "10, 25, 40" #@param {type:"string"}
if RUN_THRESHOLD_SWEEP:
    processed, _ = preprocess(input_image, cfg.remove_background, cfg.foreground_ratio)
    sc, _ = encode(processed)
    for th in [float(x) for x in THRESHOLDS.split(",")]:
        m = postprocess(extract_mesh(sc, cfg.mc_resolution, th, cfg.vertex_colors), cfg)
        s = mesh_stats(m)
        print(f"threshold={th}: faces={s['faces']}, watertight={s['watertight']}, components={s['components']}")
        show_plotly(orient_for_viewers(m), max_faces=30000, title=f"threshold={th}")
else:
    print("Tick RUN_THRESHOLD_SWEEP to execute.")

## 8 · Interactive web demo (Gradio)

In [ ]:
#@title 8.1 Launch Gradio app { display-mode: "form" }
LAUNCH_GRADIO = False #@param {type:"boolean"}
if LAUNCH_GRADIO:
    import gradio as gr
    def _fn(img, rm_bg, ratio, res, thr, smooth, decimate):
        c = replace(cfg, remove_background=rm_bg, foreground_ratio=float(ratio), mc_resolution=int(res),
                    mc_threshold=float(thr), taubin_smooth_iters=int(smooth), decimate_to_faces=int(decimate),
                    render_views=0, bake_texture=False)
        r = run_pipeline(img, c, name="gradio")
        return r["processed"], r["files"].get("glb")
    demo = gr.Interface(
        fn=_fn,
        inputs=[gr.Image(type="pil", image_mode="RGBA", label="Input image"),
                gr.Checkbox(True, label="Remove background"),
                gr.Slider(0.5, 1.0, 0.85, step=0.01, label="Foreground ratio"),
                gr.Slider(64, 512, 256, step=32, label="MC resolution"),
                gr.Slider(0, 60, 25, step=1, label="MC threshold"),
                gr.Slider(0, 30, 0, step=1, label="Taubin smoothing iterations"),
                gr.Number(0, label="Decimate to N faces (0 = off)")],
        outputs=[gr.Image(label="Processed"), gr.Model3D(label="3D model", clear_color=[0, 0, 0, 0])],
        title="Single Image → 3D", allow_flagging="never")
    demo.launch(share=True, debug=False)
else:
    print("Tick LAUNCH_GRADIO to start the demo (prints a public link).")

## 9 · Package and download your results

In [ ]:
#@title 9.1 Zip the last run (or the whole output folder) { display-mode: "form" }
ZIP_WHOLE_OUTPUT_DIR = False #@param {type:"boolean"}
src = Path(cfg.output_dir) if ZIP_WHOLE_OUTPUT_DIR else Path(result["out_dir"])
zip_path = Path(f"{src.name}.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for p in src.rglob("*"):
        if p.is_file(): z.write(p, p.relative_to(src.parent))
print("Created", zip_path, f"({zip_path.stat().st_size/1e6:.1f} MB)")
try:
    from google.colab import files as colab_files
    colab_files.download(str(zip_path))
except Exception:
    pass

## 10 · Troubleshooting

| Symptom | Fix |
|---|---|
| `torchmcubes` build fails | Notebook auto-falls back to scikit-image. Check orientation in the 3D viewer. |
| CUDA out of memory | Lower `CHUNK_SIZE` (2048/4096) and `MC_RESOLUTION` (128–192). |
| Empty mesh error | Lower `MC_THRESHOLD`, increase `FOREGROUND_RATIO`, make sure the object is segmented. |
| Holes / blobby background | Provide a clean RGBA cut-out, or enable background removal. |
| Texture baking fails | Headless OpenGL (moderngl) is unavailable; use vertex colors (default). |
| Import errors after install | `Runtime → Restart session`, then re-run from 1.2. |

## 11 · Limitations

* Feed-forward single-view reconstruction hallucinates the back of the object; fine geometry and thin structures are approximate.
* Works best on isolated, object-centric images. Scenes, humans in complex poses and transparent objects are out of distribution.
* Scale is arbitrary (meshes are normalised); metric evaluation requires scale alignment, which `chamfer_fscore` does.

## 12 · Citation & licences

If you use this notebook in research please cite the original model:

```bibtex
@article{TripoSR2024,
  title   = {TripoSR: Fast 3D Object Reconstruction from a Single Image},
  author  = {Tochilkin, Dmitry and Pankratz, David and Liu, Zexiang and Huang, Zixuan and Letts, Adam and Li, Yangguang and Liang, Ding and Laforte, Christian and Jampani, Varun and Cao, Yan-Pei},
  journal = {arXiv preprint arXiv:2403.02151},
  year    = {2024}
}
```

This notebook is MIT-licensed. TripoSR code/weights, rembg and its segmentation models have their own licences, so please review them before commercial use.